### 02A · PVAR de las 30 PCs de presión (6 h) y tres generadores
Un solo ajuste por número de armónicos (`h0`, `h1`, `h2`, BIC → p = 10, n_harm = 1). De `var_h1` salen tres generadores que solo se diferencian en los choques que se le pasan:
- `h1`: gaussianos (Cholesky de Σ mensual).
- `h1b`: bootstrap de choques sueltos del mismo mes.
- `h1bb24`: bootstrap por bloques de 24 pasos (6 días ≈ vida de una borrasca), mismo mes y misma hora de inicio. Recupera la huella de la dinámica no lineal que queda en los residuos (test_small_area_02F).

Mismas semillas en los tres. `h0` y `h2` (gaussianos, 20 sims) solo para comparar armónicos en el 02B.

In [7]:
import pandas as pd
import numpy as np
import json
from utils.tv_var.harmonic_standardizer import HarmonicStandardizer
from utils.tv_var.periodic_var import PeriodicVAR, select_order

import matplotlib.pyplot as plt

pcs_sel_6h = pd.read_csv('outputs/pca/pcs_msl_30_6h.csv', index_col=0, parse_dates=True)

#### Quitar media y desviación estacionales (la estacionalidad va aquí, no en los coeficientes)
harm_std_pcs = HarmonicStandardizer(n_harmonics=3)
harm_std_pcs.fit(pcs_sel_6h)
pcs_sel_6h_z = harm_std_pcs.transform(pcs_sel_6h)
print(pcs_sel_6h_z.shape)

(126864, 30)


In [8]:
bic_0b = select_order(pcs_sel_6h_z, p_values=[6, 8, 10, 12, 16], n_harm=0, freq="6h")
bic_0b

,nobs,loglike,n_params,bic,best
p,,,,,
6,126848,-1.946858e+06,11010,4.023092e+06,False
8,126848,-1.933072e+06,12810,4.016670e+06,False
10,126848,-1.917246e+06,14610,4.006170e+06,True
12,126848,-1.912149e+06,16410,4.017127e+06,False
16,126848,-1.899414e+06,20010,4.033961e+06,False


In [9]:
var_h0 = PeriodicVAR(p=10, n_harm=0, freq="6h").fit(pcs_sel_6h_z)
var_h1 = PeriodicVAR(p=10, n_harm=1, freq="6h").fit(pcs_sel_6h_z)
var_h2 = PeriodicVAR(p=10, n_harm=2, freq="6h").fit(pcs_sel_6h_z)

pd.DataFrame({"n_harm=0": [var_h0.n_params(), var_h0.loglike(), var_h0.bic()],
              "n_harm=1": [var_h1.n_params(), var_h1.loglike(), var_h1.bic()],
              "n_harm=2": [var_h2.n_params(), var_h2.loglike(), var_h2.bic()]},
             index=["n_params", "loglike", "bic"])

,n_harm=0,n_harm=1,n_harm=2
n_params,1.461000e+04,3.267000e+04,5.073000e+04
loglike,-1.917347e+06,-1.795356e+06,-1.761767e+06
bic,4.006372e+06,3.974610e+06,4.119651e+06


In [10]:
for n, v in {"h0": var_h0, "h1": var_h1, "h2": var_h2}.items():
    print(n, v.stability())

h0 {'rho': 0.04672212737836443, 'rho_daily': 0.9916418718401774, 'rho_step': 0.9979038866279908, 'stable': True}
h1 {'rho': 0.0237695328885974, 'rho_daily': 0.9898075028761876, 'rho_step': 0.9974420779972804, 'stable': True}
h2 {'rho': 0.02765797597605177, 'rho_daily': 0.9902184518827979, 'rho_step': 0.9975455915661394, 'stable': True}


In [11]:
print(var_h2.summary())

                         Periodic VAR                         
Variables (k)                   30    msl_PC1, msl_PC2, msl_PC3, msl_PC4, msl_PC5, msl_PC6, msl_PC7, msl_PC8, msl_PC9, msl_PC10, msl_PC11, msl_PC12, msl_PC13, msl_PC14, msl_PC15, msl_PC16, msl_PC17, msl_PC18, msl_PC19, msl_PC20, msl_PC21, msl_PC22, msl_PC23, msl_PC24, msl_PC25, msl_PC26, msl_PC27, msl_PC28, msl_PC29, msl_PC30
Frequency                       6h    0 days 06:00:00
Exogenous (q)                    0    -
Lags (p)                        10
Harmonics (n_harm)               2
Noise                       gaussian / monthly
Sample                    1940-01-03 12:00 → 2026-10-31 18:00
Observations (nobs)         126854
--------------------------------------------------------------
Coefficients                 45150    (1505 per equation)
Covariance parameters         5580    (12 block(s))
Total parameters             50730
Log-likelihood               -1761766.55
BIC                           4119650.78
-----------

#### Ruido de los generadores

In [12]:
import os, pickle, time

out = "outputs/var_6h"
os.makedirs(out, exist_ok=True)
pickle.dump(harm_std_pcs, open(f"{out}/harm_std_pcs.pkl", "wb"))
for n, v in {"h0": var_h0, "h1": var_h1, "h2": var_h2}.items():
    pickle.dump(v, open(f"{out}/var_{n}.pkl", "wb"))

burn = int(pd.Timedelta(days=3*365) / pd.Timedelta("6h"))
fechas = pd.date_range("1940-01-01", "2026-12-31 18:00", freq="6h")

def ruido_bootstrap(var, fechas, seed):
    """Choques sueltos remuestreados del pool de residuos de cada mes."""
    rng = np.random.default_rng(seed)
    eps = np.empty((len(fechas), var.k_))
    for m, pool in var.resid_month_.items():
        sel = fechas.month == m
        eps[sel] = pool[rng.integers(len(pool), size=sel.sum())]
    return pd.DataFrame(eps, index=fechas, columns=var.columns_)

def inicios_bloques(var, L):
    """Posiciones donde empieza un bloque de L residuos consecutivos, por (mes, hora) de inicio."""
    idx = var.resid_.index
    pos = np.arange(len(idx) - L + 1)
    pos = pos[(idx[pos + L - 1] - idx[pos]) == (L - 1) * pd.Timedelta("6h")]
    return {(m, h): pos[(idx.month[pos] == m) & (idx.hour[pos] == h)] for m in range(1, 13) for h in [0, 6, 12, 18]}

def ruido_bloques(var, fechas, seed, L, inicios):
    """Bloques de L choques consecutivos reales, empezando en el mismo mes y hora."""
    rng = np.random.default_rng(seed)
    R = var.resid_.values
    eps = np.empty((len(fechas), var.k_))
    for i0 in range(0, len(fechas), L):
        t = fechas[i0]
        pool = inicios[(t.month, t.hour)]
        s = pool[rng.integers(len(pool))]
        n = min(L, len(fechas) - i0)
        eps[i0:i0 + n] = R[s:s + n]
    return pd.DataFrame(eps, index=fechas, columns=var.columns_)

#### Simulaciones (las que ya existen se saltan)

In [13]:
L = 24
inicios = inicios_bloques(var_h1, L)

GENERADORES = {                                     # nombre: (modelo, función de choques o None = gaussiano, nº sims)
    "h0":     (var_h0, None, 20),
    "h2":     (var_h2, None, 20),
    "h1":     (var_h1, None, 100),
    "h1b":    (var_h1, lambda s: ruido_bootstrap(var_h1, fechas, s), 100),
    "h1bb24": (var_h1, lambda s: ruido_bloques(var_h1, fechas, s, L, inicios), 100),
}
CORRER = ["h0", "h2", "h1", "h1b", "h1bb24"]

for nombre in CORRER:
    var, choques, n_sims = GENERADORES[nombre]
    sim_dir = f"{out}/simulaciones_{nombre}"
    os.makedirs(sim_dir, exist_ok=True)
    for i in range(n_sims):
        f_sim = f"{sim_dir}/sim_{i:03d}.parquet"
        if os.path.exists(f_sim):
            continue
        t = time.time()
        if choques is None:
            sim_z = var.simulate(n_years=87, start="1940-01-01", seed=i, burn=burn)
        else:
            sim_z = var.simulate(noise=choques(i), burn=burn)
        harm_std_pcs.inverse_transform(sim_z).astype("float32").to_parquet(f_sim)
        print(f"{nombre} sim {i:03d}  ({time.time() - t:.0f} s)")

h0 sim 000  (3 s)
h0 sim 001  (2 s)
h0 sim 002  (2 s)
h0 sim 003  (2 s)
h0 sim 004  (2 s)
h0 sim 005  (2 s)
h0 sim 006  (2 s)
h0 sim 007  (2 s)
h0 sim 008  (2 s)
h0 sim 009  (2 s)
h0 sim 010  (2 s)
h0 sim 011  (2 s)
h0 sim 012  (2 s)
h0 sim 013  (2 s)
h0 sim 014  (2 s)
h0 sim 015  (2 s)
h0 sim 016  (2 s)
h0 sim 017  (2 s)
h0 sim 018  (2 s)
h0 sim 019  (2 s)
h2 sim 000  (2 s)
h2 sim 001  (2 s)
h2 sim 002  (2 s)
h2 sim 003  (2 s)
h2 sim 004  (2 s)
h2 sim 005  (2 s)
h2 sim 006  (2 s)
h2 sim 007  (2 s)
h2 sim 008  (2 s)
h2 sim 009  (2 s)
h2 sim 010  (2 s)
h2 sim 011  (2 s)
h2 sim 012  (2 s)
h2 sim 013  (2 s)
h2 sim 014  (2 s)
h2 sim 015  (2 s)
h2 sim 016  (2 s)
h2 sim 017  (2 s)
h2 sim 018  (2 s)
h2 sim 019  (2 s)
h1 sim 000  (2 s)
h1 sim 001  (2 s)
h1 sim 002  (2 s)
h1 sim 003  (2 s)
h1 sim 004  (2 s)
h1 sim 005  (2 s)
h1 sim 006  (2 s)
h1 sim 007  (2 s)
h1 sim 008  (2 s)
h1 sim 009  (2 s)
h1 sim 010  (2 s)
h1 sim 011  (2 s)
h1 sim 012  (2 s)
h1 sim 013  (2 s)
h1 sim 014  (2 s)
h1 sim 015